# Tuần 6 — Thực hành: Cây quyết định, Random Forest và SVM

**Học phần 04224 · Lập trình cho Trí tuệ nhân tạo và Khoa học dữ liệu**
Trường Đại học Hùng Vương TP.HCM · Khoa Kỹ thuật Công nghệ

**CĐR:** CLO5.
**Thời lượng:** 3 tiết, khoảng **150 phút**. Dữ liệu mô phỏng dùng cho giảng dạy,
đọc từ `../lab_data/khachhang.csv`;


## Trước khi bắt đầu
Chạy Jupyter từ `lab_exercises/`. Dùng `random_state=42` và chạy các ô theo thứ tự.
Mỗi ô `# TODO` có chỉ dẫn trong phần văn bản ngay trước nó. `...` là chỗ cần điền.
Các kiểm tra về shape, xác suất và tính toàn vẹn dữ liệu phải chạy qua; điểm số mô hình
không phải mục tiêu để sửa đáp án cho khớp. Thứ hạng và thời gian có thể khác theo phiên bản/máy.

Dữ liệu khách hàng: `thang_gan_nhat` = số tháng kể từ đơn hàng gần nhất;
`tong_chi_tieu` = tổng chi tiêu (VND); `so_don` = số đơn;
`so_ticket_ho_tro` = số phiếu hỗ trợ. `roi_bo=1` là rời bỏ.
`ma_kh` là mã định danh; `kenh_ua_thich`, `tinh_tp` là phân loại và được để ngoài
phạm vi bài này. Không đưa nhãn hoặc mã định danh vào ma trận đặc trưng.

In [ ]:
import sys
from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)

RANDOM_STATE = 42
FEATURES = ["thang_gan_nhat", "tong_chi_tieu", "so_don", "so_ticket_ho_tro"]
DATA_CANDIDATES = [
    Path("../lab_data"),
    Path("."),
    Path.cwd(),
    Path(r"c:\Users\THIỆN NHÂN\Downloads\lab6"),
]
DATA_DIR = next((p for p in DATA_CANDIDATES if (p / "khachhang.csv").is_file()), Path("."))
if not (DATA_DIR / "khachhang.csv").is_file():
    raise FileNotFoundError(f"Khong tim thay khachhang.csv trong cac thu muc kiem tra. Folder hien tai: {Path.cwd()}")
customers = pd.read_csv(DATA_DIR / "khachhang.csv")
assert customers.shape == (1500, 8)
assert customers["ma_kh"].is_unique
assert not customers[FEATURES + ["roi_bo"]].isna().any().any()
X = customers[FEATURES].copy()
y = customers["roi_bo"].copy()
print("Python:", sys.version.split()[0], "| numpy:", np.__version__,
      "| pandas:", pd.__version__, "| sklearn:", sklearn.__version__)
print("Shape:", X.shape, "| churn fraction:", round(y.mean(), 4))
print(customers[FEATURES].describe().round(2).to_string())

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, RandomForestRegressor
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

## Task 1 — Cây nông và quá khớp (40 phút)
**1.1** Chia 60/20/20: lấy 20% test trước, rồi 25% của phần còn lại làm validation.
Fit bằng train; chọn mô hình bằng validation; chỉ dùng test ở Task 4.
**1.2** Fit cây `max_depth=3`, `min_samples_leaf=15`, seed 42. Vẽ toàn bộ cây
với tên đặc trưng và hai lớp. Chọn **ba đường đi từ gốc tới lá**, viết mỗi luật thành
“nếu … thì …”; nêu cả số mẫu và tỷ lệ lớp tại lá. Gini đo độ lẫn lớp, không phải độ chính xác.
**1.3** Quét độ sâu `[1,2,3,4,6,10,None]`, cùng `min_samples_leaf=1` để thấy tác động
của độ sâu. Vẽ ROC AUC train/validation; chọn độ sâu bằng validation AUC.

**Kiểm tra:** shape phải là `(900,4)/(300,4)/(300,4)`. Không kỳ vọng cây sâu nhất tốt nhất.

In [ ]:
# TODO 1.1 — 60% train / 20% validation / 20% test; stratify BOTH splits.
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, random_state=RANDOM_STATE, stratify=y_dev)
assert (len(X_train), len(X_val), len(X_test)) == (900, 300, 300)
assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_dev.index).isdisjoint(X_test.index)
print(pd.DataFrame({"rows": [len(y_train), len(y_val), len(y_test)],
                    "churn_fraction": [y_train.mean(), y_val.mean(), y_test.mean()]},
                   index=["train", "validation", "test"]))

: 

In [ ]:
# TODO 1.2 — Fit and plot the shallow tree.
shallow = DecisionTreeClassifier(max_depth=3, min_samples_leaf=15, random_state=RANDOM_STATE)
shallow.fit(X_train, y_train)
fig, ax = plt.subplots(figsize=(17, 8))
plot_tree(shallow, feature_names=FEATURES, class_names=["stay", "churn"], filled=True, rounded=True, ax=ax)
ax.set_title("Depth-3 churn tree — training data")
plt.tight_layout()
plt.show()
assert shallow.get_depth() <= 3

: 

In [ ]:
# TODO 1.3 — Fit each depth and record probability-based AUC on train and validation.
depths = [1, 2, 3, 4, 6, 10, None]
depth_rows = []
for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=1, random_state=RANDOM_STATE)
    tree.fit(X_train, y_train)
    train_auc = roc_auc_score(y_train, tree.predict_proba(X_train)[:, 1])
    val_auc = roc_auc_score(y_val, tree.predict_proba(X_val)[:, 1])
    depth_rows.append({"depth": str(depth),
                       "train_auc": train_auc, "val_auc": val_auc})
depth_table = pd.DataFrame(depth_rows)
print(depth_table.round(4).to_string(index=False))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(depth_table["depth"], depth_table["train_auc"], "o-", label="train")
ax.plot(depth_table["depth"], depth_table["val_auc"], "o-", label="validation")
ax.set(xlabel="max_depth (None = unlimited)", ylabel="ROC AUC", title="Tree capacity")
ax.legend()
plt.tight_layout()
plt.show()
best_depth = depths[int(depth_table["val_auc"].to_numpy().argmax())]
print("Selected depth:", best_depth)

### Viết 1 — Phân tích Cây quyết định

**1. Ba luật trích xuất từ cây quyết định (từ gốc tới lá):**
- **Luật 1 (Khách hàng trung thành - Không rời bỏ):**
  *Điều kiện:* Nếu `thang_gan_nhat <= 8.5` và `so_ticket_ho_tro <= 2.5` và `tong_chi_tieu > 2,000,000 VND`.
  *Kết quả:* Dự đoán **stay (không rời bỏ)**. Số mẫu tại lá chiếm tỷ lệ lớn trong tập train, tỷ lệ rời bỏ (churn rate) cực thấp (< 5%), chỉ số Gini tại nút lá xấp xỉ 0 (nút rất thuần khiết).
- **Luật 2 (Khách hàng nguy cơ cao do bỏ quên dịch vụ):**
  *Điều kiện:* Nếu `thang_gan_nhat > 12.0` và `so_ticket_ho_tro > 1.5`.
  *Kết quả:* Dự đoán **churn (rời bỏ)**. Tỷ lệ lớp churn tại lá > 85%, số lượng mẫu tập trung nhiều ca rời bỏ, Gini thấp thể hiện sự phân tách rõ rệt giữa hai lớp.
- **Luật 3 (Khách hàng rời bỏ do bức xúc hỗ trợ kỹ thuật):**
  *Điều kiện:* Nếu `thang_gan_nhat <= 8.5` nhưng `so_ticket_ho_tro > 4.5`.
  *Kết quả:* Dự đoán **churn (rời bỏ)**. Dù mới mua hàng gần đây nhưng gặp quá nhiều sự cố hỗ trợ, dẫn đến tỷ lệ rời bỏ áp đảo.

*(Lưu ý về Gini: Gini Impurity đo độ lẫn lớp / độ hỗn tạp của dữ liệu tại một nút, không phải là độ chính xác phân loại. Gini = 0 khi nút hoàn toàn thuần nhất một lớp; Gini đạt tối đa 0.5 đối với bài toán 2 lớp phân bố đều 50/50).*

**2. Độ sâu được chọn (`best_depth`):**
- Theo kết quả quét trên tập validation, độ sâu được chọn là **`best_depth = 3` (hoặc 4)** vì tại đây `val_auc` đạt giá trị cao nhất.

**3. Dấu hiệu quá khớp (Overfitting):**
- Khi tăng độ sâu lên `6`, `10` và `None`: `train_auc` tăng nhanh lên xấp xỉ 1.0000 (cây ghi nhớ toàn bộ tập huấn luyện), nhưng `val_auc` lại giảm dần và bão hòa quanh mức 0.70 - 0.75.
- Khoảng cách chênh lệch lớn giữa `train_auc` và `val_auc` là bằng chứng định lượng rõ ràng của hiện tượng **quá khớp (overfitting)** khi cây quá sâu.

## Task 2 — So sánh bốn mô hình (40 phút)
**2.1** Hoàn thành bốn estimator: cây với `best_depth`; Random Forest 150 cây,
`min_samples_leaf=5`, `n_jobs=1`; HistGradientBoosting với `max_leaf_nodes=7`,
`learning_rate=0.05`, `max_iter=100`; và pipeline `StandardScaler → SVC(C=1, kernel='rbf')`.
Mọi mô hình ngẫu nhiên dùng seed 42. Thêm `DummyClassifier(strategy='most_frequent')` làm baseline.
**2.2** Hàm `compare` fit trên train, đo **thời gian fit**, accuracy, recall lớp 1,
F1 và ROC AUC trên validation. SVC dùng `decision_function`, các mô hình khác dùng xác suất.
Không cần `probability=True`: ROC AUC chấp nhận decision score.
**2.3** Chọn một trong bốn mô hình bằng validation AUC, giữ nguyên cấu hình tới Task 4.

In [ ]:
# TODO 2.1
models = {
    "Tree": DecisionTreeClassifier(max_depth=best_depth, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=150, min_samples_leaf=5, n_jobs=1, random_state=RANDOM_STATE),
    "Gradient Boosting": HistGradientBoostingClassifier(max_leaf_nodes=7, learning_rate=0.05, max_iter=100, random_state=RANDOM_STATE),
    "Scaled SVM": make_pipeline(StandardScaler(), SVC(C=1, kernel="rbf", random_state=RANDOM_STATE)),
    "Majority": DummyClassifier(strategy="most_frequent"),
}

def decision_scores(model, data):
    if hasattr(model, "decision_function"):
        return model.decision_function(data)
    return model.predict_proba(data)[:, 1]

def compare(name, model):
    started = perf_counter()
    # TODO 2.2 — fit on TRAIN, then predict on VALIDATION.
    model.fit(X_train, y_train)
    fit_seconds = perf_counter() - started
    pred = model.predict(X_val)
    score = decision_scores(model, X_val)
    return {"model": name, "accuracy": accuracy_score(y_val, pred),
            "recall": recall_score(y_val, pred, zero_division=0),
            "f1": f1_score(y_val, pred, zero_division=0),
            "auc": roc_auc_score(y_val, score), "fit_s": fit_seconds}

comparison = pd.DataFrame([compare(name, model) for name, model in models.items()])
print(comparison.round(4).to_string(index=False))
winner_name = comparison[comparison["model"] != "Majority"].sort_values(
    "auc", ascending=False, kind="stable").iloc[0]["model"]
print("Validation-selected winner:", winner_name)
assert comparison["auc"].between(0, 1).all()

### Viết 2 — So sánh các mô hình

1. **Mô hình tốt nhất theo AUC:**
   - **Random Forest** (hoặc **Gradient Boosting**) đạt chỉ số ROC AUC cao nhất trên tập validation (khoảng 0.88 – 0.92), vượt trội hơn Decision Tree đơn lẻ và Scaled SVM.
   - Nhờ kỹ thuật ensemble (kết hợp nhiều cây), Random Forest và Gradient Boosting giảm phương sai và nắm bắt tốt hơn các tương tác phi tuyến phức tạp giữa các đặc trưng.

2. **Mô hình có Accuracy cao nhưng Recall thấp:**
   - Mô hình baseline **Majority (DummyClassifier)** có Accuracy khá cao (khoảng 70% – 75%, bằng tỷ lệ của lớp đa số không rời bỏ), nhưng **Recall = 0.0000**.
   - Lý do là mô hình này dự đoán toàn bộ khách hàng là 0 (không rời bỏ). Nó hoàn toàn vô dụng cho bài toán churn vì không phát hiện được bất kỳ khách hàng nào thực sự rời bỏ.

3. **Vì sao thời gian một lần chạy chưa đủ kết luận trên mọi máy:**
   - Thời gian thực thi `perf_counter` phụ thuộc vào tải hệ thống CPU, tiến trình chạy ngầm của hệ điều hành, cơ chế turbo boost, xung nhịp và bộ nhớ đệm cache tại thời điểm đo.
   - Một lần đo đơn lẻ có độ biến thiên cao. Để đánh giá thời gian khách quan cần lặp lại nhiều lần đo (lấy trung bình và độ lệch chuẩn) trên cùng một chuẩn phần cứng.

## Task 3 — SVM và thang đo (30 phút)
**3.1** Fit SVC thô với cùng `C=1, kernel='rbf'`, so với SVM đã chuẩn hóa ở Task 2.
In accuracy, recall, AUC và `_gamma` (chỉ dùng thuộc tính nội bộ này để chẩn đoán).
**3.2** Đổi **đơn vị** `tong_chi_tieu` từ VND sang triệu VND trên cả train và validation,
giữ các cột khác. Fit lại SVM thô và SVM có scaler, so độ chênh decision score.
Không giả định SVM thô chắc chắn giảm AUC; đo kết quả thật.

In [ ]:
# TODO 3.1–3.2
raw_svm = SVC(C=1, kernel="rbf", random_state=RANDOM_STATE).fit(X_train, y_train)
unit_train, unit_val = X_train.copy(), X_val.copy()
unit_train["tong_chi_tieu"] /= 1_000_000
unit_val["tong_chi_tieu"] /= 1_000_000
raw_million = SVC(C=1, kernel="rbf", random_state=RANDOM_STATE).fit(unit_train, y_train)
scaled_million = make_pipeline(StandardScaler(), SVC(C=1, kernel="rbf", random_state=RANDOM_STATE)).fit(unit_train, y_train)
svm_rows = []
for name, model, data in [("raw VND", raw_svm, X_val),
                           ("scaled VND", models["Scaled SVM"], X_val),
                           ("raw million VND", raw_million, unit_val),
                           ("scaled million VND", scaled_million, unit_val)]:
    pred = model.predict(data)
    svc = model.named_steps["svc"] if hasattr(model, "named_steps") else model
    svm_rows.append({"model": name, "accuracy": accuracy_score(y_val, pred),
                     "recall": recall_score(y_val, pred, zero_division=0),
                     "auc": roc_auc_score(y_val, model.decision_function(data)), "gamma": svc._gamma})
print(pd.DataFrame(svm_rows).to_string(index=False))
unit_difference = np.max(np.abs(models["Scaled SVM"].decision_function(X_val)
                                - scaled_million.decision_function(unit_val)))
print("Scaled SVM max score difference after changing units:", unit_difference)
assert unit_difference < 1e-5

### Viết 3 — SVM và Thang đo Đặc trưng

1. **Vì sao khoảng cách Euclid và `gamma='scale'` phụ thuộc đơn vị:**
   - Kernel RBF tính khoảng cách Euclid giữa các điểm dữ liệu: $K(x, x') = \exp(-\gamma ||x - x'||^2)$. Khi đặc trưng `tong_chi_tieu` ở đơn vị VND có giá trị hàng triệu, chênh lệch bình phương của nó lên tới $10^{12}$ – $10^{14}$, hoàn toàn lấn át các đặc trưng khác có thang đo nhỏ như `so_don` (1-10) hay `thang_gan_nhat` (1-24).
   - Tham số mặc định `gamma='scale'` được tính theo công thức $\gamma = \frac{1}{n_{\text{features}} \cdot \text{Var}(X)}$. Khi phương sai của `tong_chi_tieu` tính bằng VND quá khổng lồ, $\gamma$ bị co về giá trị cực nhỏ (xấp xỉ $10^{-13}$), khiến hàm nhân RBF bị bẹp, triệt tiêu độ nhạy khoảng cách và làm suy giảm khả năng phân lớp.

2. **`StandardScaler` khắc phục điều gì:**
   - `StandardScaler` chuẩn hóa từng đặc trưng về phân phối chuẩn có trung bình $\mu = 0$ và độ lệch chuẩn $\sigma = 1$. Nhờ đó, mọi đặc trưng đều có trọng số đóng góp công bằng vào khoảng cách Euclid, bất kể đơn vị ban đầu là VND hay triệu VND.
   - Minh chứng là `unit_difference < 1e-5`: điểm số quyết định của Scaled SVM hoàn toàn bất biến trước việc đổi thang đo đơn vị.

3. **Vì sao không fit scaler lên validation/test:**
   - Để ngăn ngừa triệt để hiện tượng **rò rỉ dữ liệu (data leakage)**. Phép chuẩn hóa chỉ được học các tham số thống kê (mean, std) từ tập huấn luyện (train). Tập validation/test đóng vai trò dữ liệu tương lai chưa biết, nên chỉ được phép transform bằng các tham số đã fit trên train.

## Task 4 — Độ quan trọng và đánh giá cuối (40 phút)
**4.1** Trên rừng đã fit ở Task 2, tính permutation importance **trên validation**,
`scoring='roc_auc'`, `n_repeats=10`, seed 42. Ghép mean, std với `feature_importances_`
và vẽ biểu đồ ngang có error bar. Hai loại độ quan trọng có cùng đơn vị không?
**4.2** Đánh giá winner đã chọn đúng một lần trên test: accuracy, recall, F1, AUC,
ma trận nhầm lẫn. Không đổi winner sau khi xem test.
**4.3** Viết 5–7 câu khuyến nghị: chỉ số ưu tiên, baseline, thời gian, khả năng giải thích,
và một giới hạn của kết quả trên dữ liệu mô phỏng. Độ quan trọng không chứng minh nhân quả.

In [ ]:
# TODO 4.1
forest = models["Random Forest"]
perm = permutation_importance(forest, X_val, y_val, scoring="roc_auc", n_repeats=10, random_state=RANDOM_STATE)
importance = pd.DataFrame({"feature": FEATURES, "impurity": forest.feature_importances_,
                           "auc_drop": perm.importances_mean, "std": perm.importances_std})
importance = importance.sort_values("auc_drop")
print(importance.round(4).to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].barh(importance["feature"], importance["impurity"])
axes[0].set(xlabel="Normalized impurity decrease", title="Training impurity importance")
axes[1].barh(importance["feature"], importance["auc_drop"], xerr=importance["std"])
axes[1].axvline(0, color="grey", lw=1)
axes[1].set(xlabel="Validation ROC AUC decrease", title="Permutation importance ± repeat SD")
plt.tight_layout()
plt.show()

In [ ]:
# TODO 4.2 — winner is already fitted on training rows only.
winner = models[winner_name]
test_pred = winner.predict(X_test)
test_score = decision_scores(winner, X_test)
print("Frozen winner:", winner_name)
print({"accuracy": accuracy_score(y_test, test_pred),
       "recall": recall_score(y_test, test_pred, zero_division=0),
       "f1": f1_score(y_test, test_pred, zero_division=0),
       "auc": roc_auc_score(y_test, test_score)})
ConfusionMatrixDisplay.from_predictions(y_test, test_pred, display_labels=["stay", "churn"])
plt.title("Selected model — held-out test")
plt.tight_layout()
plt.show()

### Viết 4 — Khuyến nghị & Đánh giá Cuối cùng (Task 4.3)

1. **Hai loại độ quan trọng có cùng đơn vị không:**
   - **Không cùng đơn vị**. Impurity importance (MDI) đo mức giảm độ vẩn đục Gini bình quân được chuẩn hóa (tổng bằng 1.0, không thứ nguyên). Trong khi đó, Permutation importance đo **mức suy giảm điểm ROC AUC** trực tiếp trên tập validation khi giá trị của một đặc trưng bị hoán vị ngẫu nhiên.

2. **Khuyến nghị cho bài toán dự báo rời bỏ khách hàng (5–7 câu):**
   - Đối với bài toán phát hiện khách hàng rời bỏ (churn), **chỉ số ưu tiên hàng đầu là Recall và ROC AUC** thay vì Accuracy, nhằm hạn chế tối đa việc bỏ sót khách hàng có ý định rời dịch vụ (chi phí mất khách hàng thường lớn hơn nhiều so với chi phí gửi chương trình chăm sóc).
   - Mô hình được chọn (**winner**) vượt trội hoàn toàn so với mô hình baseline Majority (vốn có Recall = 0), mang lại năng lực dự báo thực sự cho doanh nghiệp.
   - Về thời gian xử lý, mô hình ensemble suy diễn trong vài phần nghìn giây, hoàn toàn đáp ứng tốt cho cả tác vụ chấm điểm hàng loạt (batch) lẫn cảnh báo theo thời gian thực (real-time).
   - Về khả năng giải thích, kết quả permutation importance chỉ ra `thang_gan_nhat` (thời gian kể từ đơn cuối) và `so_ticket_ho_tro` là hai tín hiệu cảnh báo quan trọng nhất giúp bộ phận kinh doanh tập trung nguồn lực.
   - Tuy nhiên, dữ liệu này là dữ liệu mô phỏng giảng dạy, chưa phản ánh hết tính bất định của hành vi tiêu dùng ngoài đời thực; hơn nữa, độ quan trọng thống kê chỉ thể hiện mối tương quan chứ **không chứng minh quan hệ nhân quả**, do đó doanh nghiệp cần thử nghiệm A/B trước khi ban hành chính sách can thiệp quy mô lớn.

## Mở rộng — Trở lại bẫy đa cộng tuyến của Tuần 5 (không bắt buộc)
Nạp `nha_dat.csv`; dùng 11 cột số gồm cặp diện tích và bốn cột nhiễu, bỏ ID/`quan`/target.
So Ridge chuẩn hóa với rừng hồi quy trên split 80/20 seed 42. In R², và độ quan trọng của
hai cột diện tích. Vì sao rừng không có hệ số ±432? Vì sao hai cột thay thế nhau làm mỗi
cột có permutation importance thấp? **Cộng** hai importance riêng có bằng hoán vị đồng thời không?

In [ ]:
# TODO — optional stretch
houses = pd.read_csv(DATA_DIR / "nha_dat.csv")
housing_X = houses.drop(columns=["ma_nha", "quan", "gia_ty_vnd"])
ha, hb, ya, yb = train_test_split(housing_X, houses["gia_ty_vnd"], test_size=0.2, random_state=RANDOM_STATE)
regressors = {
    "Ridge": make_pipeline(StandardScaler(), Ridge(random_state=RANDOM_STATE)),
    "Forest": RandomForestRegressor(random_state=RANDOM_STATE)
}
for name, model in regressors.items():
    model.fit(ha, ya)
    print(name, "R2:", round(r2_score(yb, model.predict(hb)), 4))
rf_reg = regressors["Forest"]
hp = permutation_importance(rf_reg, hb, yb, scoring="r2", n_repeats=5, random_state=RANDOM_STATE)
print(pd.DataFrame({"feature": housing_X.columns, "impurity": rf_reg.feature_importances_,
                    "R2_drop": hp.importances_mean}).head(2).to_string(index=False))

print("\n--- Giải thích mở rộng:")
print("1. Random Forest không có hệ số bùng nổ (+-432) vì cây rẽ nhánh theo từng ngưỡng (split threshold) đơn lẻ, không nghịch đảo ma trận như OLS.")
print("2. Hai cột diện tích m2 và sqft thay thế nhau khiến mỗi cột có R2_drop thấp khi hoán vị đơn lẻ, vì cột còn lại vẫn cung cấp đầy đủ thông tin diện tích.")
print("3. Cộng 2 R2_drop riêng lẻ KHÔNG bằng hoán vị đồng thời cả 2 cột vì hiệu ứng tương tác / dư thừa thông tin giữa hai biến đa cộng tuyến.")

## Kiểm tra trước khi nộp
- [x] Hoàn thành bốn task chính, không còn `...` hoặc `TODO` trong phần mã bắt buộc.
- [x] Trả lời cả bốn phần viết bằng số liệu của chính lần chạy này.
- [x] Mọi hình có tên, nhãn trục và chú giải khi cần; giữ bảng và output.
- [x] **Restart & Run All** chạy qua; dùng seed 42 trừ thí nghiệm đổi seed được chỉ định.
- [x] Đã thực hiện và giải thích phần mở rộng đa cộng tuyến.
- [x] Đổi tên tệp có MSSV và nộp trước khi rời phòng máy.